# Titanic Dataset: EDA & Linear Regression Tutorial
### Adapted from [UW-Madison ML-X Nexus — Titanic Dataset](https://colab.research.google.com/github/UW-Madison-DataScience/ML-X-Nexus/blob/main/Learn/Notebooks/Titanic-Dataset.ipynb)

### Categories
- Notebooks
- EDA
- Tabular
- Linear Regression
- Code-along

### Data & Problem Intro
The Titanic dataset is a well-known dataset that contains information about the passengers of the Titanic ship. It includes variables such as age, gender, class, fare, and whether each passenger survived.

The problem we are exploring is **binary classification**: predicting whether a passenger survived based on their features. We use **linear regression** as an educational baseline — a common first model before introducing classification-specific algorithms.

The goal of this exploratory data analysis (EDA) is to uncover insights that can guide our modeling decisions, such as identifying important features, handling missing data, and addressing bias in the dataset.

## Step 0: Looking Up Each Feature
Before diving into the analysis, it's important to understand what each feature in the dataset represents.

### Titanic Dataset Features (Kaggle Version):
- **PassengerId**: Unique passenger identifier.
- **Survived**: Whether the passenger survived (0 = No, 1 = Yes). **This is our target variable.**
- **Pclass**: Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd). Proxy for socio-economic status.
- **Name**: Full name of the passenger (contains title information).
- **Sex**: Gender of the passenger.
- **Age**: Age of the passenger in years. Some values are missing.
- **SibSp**: Number of siblings/spouses aboard the Titanic.
- **Parch**: Number of parents/children aboard the Titanic.
- **Ticket**: Ticket number (alphanumeric, not directly useful).
- **Fare**: Passenger fare paid.
- **Cabin**: Cabin number (missing for most passengers).
- **Embarked**: Port of embarkation (C = Cherbourg; Q = Queenstown; S = Southampton).

## Step 1: Visualizing Data in Its Rawest Form
Let's take a look at a small sample of the dataset to understand the raw data we're working with. This gives us a chance to spot obvious issues or patterns.

### Import libraries and load dataset

In [ ]:
# Install missingno if not already available
try:
    import missingno
except ImportError:
    import subprocess

    subprocess.run(["pip", "install", "missingno", "-q"])

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import warnings

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

print("Libraries loaded successfully.")

In [ ]:
DATA_DIR = "data/titanic"

df = pd.read_csv(f"{DATA_DIR}/training.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"Training set : {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Test set     : {test_df.shape[0]} rows, {test_df.shape[1]} columns")

# Display the first few rows of the dataset
df.head()

In [ ]:
# Show a small random sample of the data
print("Sample of 30 passengers:")
df.sample(30)

### Insights
- We see various features such as age, sex, class, fare, and whether the passenger survived.
- This helps us get a quick overview of what kind of data we're working with, including potential issues like missing values.
- Some NaNs are clearly visible in **Age**, **Cabin**, and **Embarked**.
- **PassengerId**, **Name**, **Ticket**, and **Cabin** are unlikely to be directly useful as model inputs.

#### Remove non-informative columns
Let's drop columns that don't carry useful signal for modeling. We'll extract a `Title` feature from `Name` before dropping it.

In [ ]:
# Extract Title from Name before dropping it
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
test_df["Title"] = test_df["Name"].str.extract(r",\s*([^\.]+)\.")

print("Titles found in training set:")
print(df["Title"].value_counts())

# Drop columns with no direct modeling value
df.drop(["PassengerId", "Name", "Ticket"], axis=1, inplace=True)
print("\nColumns after dropping non-informative fields:")
print(df.columns.tolist())

## Step 2: Check Data Types

In [ ]:
df.dtypes

## Step 3: Basic Statistics
Now we'll summarize the numerical and categorical columns to better understand the central tendencies, variability, and potential missing data.

When exploring **numerical data**:
- **Mean vs Median**: If the mean is much higher or lower than the median, this suggests skewness, possibly due to outliers.
- **Min and Max**: Look at extremes to detect outliers or data entry errors.
- **Standard deviation**: High std means data is spread widely; low std means clustered around the mean.

When exploring **categorical data**:
- **Unique counts**: The number of distinct categories.
- **Mode**: Which category dominates.
- **Frequency distribution**: Imbalanced categories can bias the model if not handled.

In [ ]:
print("Basic statistics for all columns:")
df.describe(include="all")

### Insights
- **Age**: Mean ~30, but with missing values and extreme values (min ≈ 0.42, max = 80). Mean and median are close, suggesting a fairly symmetric distribution.
- **Fare**: Wide range (0 – 512). Mean > median, indicating right skew and the presence of outliers.
- **SibSp / Parch**: Most passengers had few or no relatives onboard (median = 0 for both).
- **Pclass**: Most passengers are in 3rd class.

In [ ]:
print("Basic statistics for categorical columns:")
df.describe(include="object")

### Insights
- More males than females in this dataset.
- Southampton (S) is the most common embarkation point.

## Step 2.1: Identifying Rare Categories
To detect rare categories, we examine the frequency distribution of each categorical column. Rare categories can introduce bias or affect model performance if not handled properly.

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    print(f"Value counts for {col}:")
    print(df[col].value_counts())
    print()

### Insights
- **Sex**: No rare categories.
- **Embarked**: 'C' and 'Q' are far less common than 'S'.
- **Title**: Many rare titles (Dr, Rev, Major, etc.). We will group these into a 'Rare' category.

## Step 3: Counting Missing Values (NaNs)
To get a better understanding of where data is missing, we'll count the number of NaN values in each column.

In [ ]:
print("Number of NaNs per column:")
print(df.isna().sum())

## Step 3.1: Visualizing Missing Data
Visualizing missing data helps us understand how much data is missing and where. This informs how we should handle missing values during preprocessing.

The **Cabin** column has a very high proportion of missing values (~77%), making it nearly unusable without substantial imputation.

In [ ]:
# Visualize missing data using missingno
msno.matrix(df)
plt.show()

msno.bar(df)
plt.show()

### Insights
- **Age** has ~20% missing values — significant but manageable with imputation.
- **Cabin** has ~77% missing — too sparse to be useful without extensive imputation.
- **Embarked** has only 2 missing values — easy to fill with the mode.

We'll remove `Cabin` and impute `Age` using median per `Title` group (a more accurate imputation than the global median).

In [ ]:
# Remove Cabin (not enough info)
df.drop("Cabin", axis=1, inplace=True)
print("Cabin column dropped.")

In [ ]:
# Explore the relationship between Title and Age (similar to 'who' in the seaborn dataset)
# First, consolidate rare titles
rare_titles = df["Title"].value_counts()[df["Title"].value_counts() < 10].index
df["Title"] = df["Title"].replace(rare_titles, "Rare")
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})

sns.boxplot(x="Title", y="Age", data=df)
plt.title("Age Distribution by Title")
plt.xticks(rotation=15)
plt.show()

### Insights
- `Master` (boys) have clearly lower ages.
- `Mr`, `Mrs`, and `Miss` show distinct but overlapping distributions.
- Imputing missing `Age` values using the median per `Title` group is more accurate than using the global median.

### Impute Age (instead of dropping rows)
Rather than discarding the ~20% of rows with missing `Age` (as done in the reference notebook), we'll impute using group medians, preserving more data.

In [ ]:
# Impute Age using median per Title group
df["Age"] = df.groupby("Title")["Age"].transform(lambda x: x.fillna(x.median()))
df["Age"].fillna(df["Age"].median(), inplace=True)

# Impute Embarked with mode
df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)

print("NaNs after imputation:")
print(df.isna().sum())

## Step 4: Identifying Outliers
Outliers can distort model performance and influence relationships between features. We'll use boxplots to identify outliers in numerical columns.

### Why Look for Outliers?
- **Age**: Extreme values (very young or very old) might influence survival predictions.
- **Fare**: We've already identified skewness — high fares could represent wealthy individuals who had better survival chances.

In [ ]:
for col in ["Age", "Fare"]:
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

### Insights
- **Age**: A few extreme values for older passengers, but the distribution is mostly reasonable.
- **Fare**: Confirms the presence of outliers — some passengers paid significantly more than the majority.

## Step 5.1: Probability Density Plot for Fare
A probability density plot (PDF) lets us visualize the overall distribution, highlighting skewness or concentration of values.

In [ ]:
sns.kdeplot(df["Fare"].dropna(), fill=True)
plt.title("Probability Density Plot for Fare")
plt.xlabel("Fare")
plt.ylabel("Density")
plt.show()

### Insights
- Strong right skew: most passengers paid lower fares, but a few paid very high fares.
- The density tapers off gradually toward the high-fare end, confirming extreme outliers.

## Step 5.2: Log Scaling for Fare
Applying a log transformation compresses the range, reducing the influence of extreme outliers while preserving relative differences. Log scaling makes highly skewed distributions more normal-like and easier for linear models to handle.

In [ ]:
# Apply log scaling (add 1 to avoid log(0))
df["log_Fare"] = df["Fare"].apply(lambda x: np.log(x + 1))

sns.kdeplot(df["log_Fare"].dropna(), fill=True)
plt.title("Probability Density Plot for Log-Scaled Fare")
plt.xlabel("Log(Fare + 1)")
plt.ylabel("Density")
plt.show()

### Insights
- After log transformation, the fare distribution is much more symmetric and closer to a normal distribution.
- This will help linear regression, which assumes a linear relationship between features and the target.

## Step 6: Exploring Correlations
We'll check for correlations between numerical features. This helps identify strongly related features that could introduce multicollinearity.

### Step 6.1: Encode Categorical Data as Numeric
Encoding categorical data lets us measure correlations across all features and prepares the data for modeling.

> `pd.get_dummies(..., drop_first=True)` one-hot encodes all categorical columns. `drop_first=True` prevents multicollinearity by removing one category per feature.

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded.head()

In [ ]:
corr_matrix = df_encoded.corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix (Numerical and One-Hot Encoded Features)")
plt.tight_layout()
plt.show()

# Show correlations with Survived specifically
print("\nTop correlations with Survived:")
corr_matrix["Survived"].drop("Survived").abs().sort_values(ascending=False).head(10)

### Insights
- **Sex_male** has the strongest negative correlation with survival — being male strongly predicted not surviving.
- **Pclass** is negatively correlated with survival — lower class meant lower survival odds.
- **Fare** (and `log_Fare`) is positively correlated with survival, largely because it's linked to `Pclass`.
- **Title** encodes both gender and social status, making it a useful composite feature.
- Weak correlations with survival overall suggest non-linear models may ultimately perform better.

## Step 6.2: Pairplot for Visualizing Pairwise Relationships
Seaborn's pairplot visualizes pairwise relationships between numerical features, colored by survival outcome.

In [ ]:
# Use a subset of features for readability
pairplot_cols = ["Survived", "Pclass", "Age", "log_Fare", "SibSp", "Parch"]
sns.pairplot(df_encoded[pairplot_cols], hue="Survived", diag_kind="kde")
plt.suptitle("Pairplot of Key Features Colored by Survival", y=1.02)
plt.show()

### Insights
- Some separation between survivors and non-survivors is visible for `Pclass` and `log_Fare`.
- Pairwise relationships are not perfectly linear, suggesting non-linear models may perform better.
- These visualizations help identify where additional feature engineering may be needed.

---

## Step 7: Feature Engineering
Based on the EDA, we derive a few additional features that may capture useful signal:

In [ ]:
def engineer_features(df):
    df = df.copy()
    # Family size and solo-travel flag
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    return df


df = engineer_features(df)
print("Feature engineering complete.")
df[["FamilySize", "IsAlone"]].head()

## Step 8: Preprocessing for Modeling

In [ ]:
TARGET = "Survived"
DROP_FOR_MODEL = ["Fare"]  # replaced by log_Fare

df_model = df.drop(columns=DROP_FOR_MODEL)

# One-hot encode all remaining categoricals
df_model_enc = pd.get_dummies(df_model, drop_first=True)

X = df_model_enc.drop(columns=[TARGET])
y = df_model_enc[TARGET]

print(f"Feature matrix shape : {X.shape}")
X.head()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)

print(f"Train size      : {X_train.shape[0]}")
print(f"Validation size : {X_val.shape[0]}")

## Step 9: Train Linear Regression Model

> **Note:** Survival is a binary variable (0/1), so logistic regression is theoretically more appropriate. Linear regression is applied here as an **educational baseline**. It treats survival as a continuous score; predictions below 0 or above 1 are possible, which is a known limitation we'll observe in the residual plot.

In [ ]:
model = LinearRegression()
model.fit(X_train_sc, y_train)

print(f"Model trained.")
print(f"Intercept : {model.intercept_:.4f}")

In [ ]:
# Feature coefficients — sorted by absolute magnitude
coef_df = pd.DataFrame({"Feature": X_train.columns, "Coefficient": model.coef_})
coef_df = coef_df.sort_values("Coefficient", key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ["#2ecc71" if c > 0 else "#e74c3c" for c in coef_df["Coefficient"]]
plt.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors, edgecolor="black")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Linear Regression — Feature Coefficients")
plt.xlabel("Coefficient Value")
plt.tight_layout()
plt.show()

## Step 10: Evaluate the Model

In [ ]:
y_pred_raw = model.predict(X_val_sc)
y_pred = (y_pred_raw >= 0.5).astype(int)  # threshold at 0.5

mse = mean_squared_error(y_val, y_pred_raw)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, y_pred_raw)
acc = accuracy_score(y_val, y_pred)

print(f"MSE                       : {mse:.4f}")
print(f"RMSE                      : {rmse:.4f}")
print(f"R²                        : {r2:.4f}")
print(f"Accuracy (threshold 0.5)  : {acc:.4f}")

In [ ]:
# 5-fold cross-validation
cv_r2 = cross_val_score(model, scaler.fit_transform(X), y, cv=5, scoring="r2")
cv_acc = cross_val_score(
    LinearRegression(), scaler.fit_transform(X), y, cv=5, scoring="accuracy"
)

print(f"5-Fold CV R² scores       : {cv_r2.round(4)}")
print(f"Mean CV R²                : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"Mean CV Accuracy          : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(
    y_val, y_pred_raw, alpha=0.5, color="steelblue", edgecolors="white", linewidths=0.4
)
axes[0].axhline(0.5, color="red", linestyle="--", label="Decision boundary (0.5)")
axes[0].set_xlabel("Actual Survived")
axes[0].set_ylabel("Predicted Score")
axes[0].set_title("Actual vs Predicted Survival Score")
axes[0].legend()

# Residual Plot
residuals = y_val - y_pred_raw
axes[1].scatter(
    y_pred_raw,
    residuals,
    alpha=0.5,
    color="darkorange",
    edgecolors="white",
    linewidths=0.4,
)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_xlabel("Predicted Score")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residual Plot")

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["Did Not Survive", "Survived"]
)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix — Validation Set")
plt.tight_layout()
plt.show()

In [ ]:
# Predicted score distribution
plt.figure(figsize=(9, 4))
sns.histplot(y_pred_raw, bins=30, kde=True, color="steelblue")
plt.axvline(0.5, color="red", linestyle="--", label="Decision boundary (0.5)")
plt.title("Distribution of Predicted Survival Scores")
plt.xlabel("Predicted Score")
plt.legend()
plt.tight_layout()
plt.show()

### Insights
- The residual plot shows **non-random patterns** — because the true outcome is binary, residuals cluster around ±1 and ±0, a structural limitation of linear regression on classification tasks.
- Predictions outside [0, 1] occur, which logistic regression would naturally prevent.
- Despite these limitations, the model achieves reasonable accuracy as a baseline.

## Step 11: Predictions on Test Set

In [ ]:
# Apply same preprocessing to test set
test_ids = test_df["PassengerId"].copy()

# Title consolidation
rare_titles_test = (
    test_df["Title"].value_counts()[test_df["Title"].value_counts() < 10].index
)
test_df["Title"] = test_df["Title"].replace(rare_titles_test, "Rare")
test_df["Title"] = test_df["Title"].replace(
    {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
)

# Drop same columns
test_df.drop(
    ["PassengerId", "Name", "Ticket", "Cabin"], axis=1, inplace=True, errors="ignore"
)

# Impute
test_df["Age"] = test_df.groupby("Title")["Age"].transform(
    lambda x: x.fillna(x.median())
)
test_df["Age"].fillna(test_df["Age"].median(), inplace=True)
test_df["Fare"].fillna(
    test_df.groupby("Pclass")["Fare"].transform("median"), inplace=True
)
test_df["Embarked"].fillna(test_df["Embarked"].mode()[0], inplace=True)

# Engineer features
test_df = engineer_features(test_df)
test_df["log_Fare"] = test_df["Fare"].apply(lambda x: np.log(x + 1))
test_df.drop(columns=["Fare"], inplace=True, errors="ignore")

# Encode
test_enc = pd.get_dummies(test_df, drop_first=True)
test_enc = test_enc.reindex(columns=X.columns, fill_value=0)

# Scale & predict
test_scaled = scaler.transform(test_enc)
test_scores = model.predict(test_scaled)
test_predictions = (test_scores >= 0.5).astype(int)

submission = pd.DataFrame({"PassengerId": test_ids, "Survived": test_predictions})
submission.to_csv(f"{DATA_DIR}/submission.csv", index=False)

print(
    f"Submission saved. Predicted survivors: {test_predictions.sum()} / {len(test_predictions)}"
)
submission.head(10)

## Conclusion: Key Takeaways & Next Steps

### Key Takeaways:
1. **Feature Engineering**: `Sex`, `Pclass`, `Title`, and `log_Fare` are the strongest predictors. `FamilySize` and `IsAlone` add modest signal.
2. **Handling Missing Data**: Imputing `Age` by `Title` group is more accurate than dropping rows or using the global median.
3. **Outliers**: Log-scaling `Fare` reduced skewness and improved the feature's behavior in the linear model.
4. **Linear Regression Limitations**: The residual plot reveals non-random structure — a fundamental consequence of applying a regression model to a binary outcome. Predictions can fall outside [0, 1].

### Inspirational Next Steps:
- **Logistic Regression**: The natural upgrade for binary classification — bounds predictions to [0, 1] and directly models the log-odds of survival.
- **Tree-Based Models**: Random forests and gradient boosting (XGBoost/LightGBM) can capture non-linear relationships and feature interactions.
- **Iterative EDA**: As you build models, revisit the EDA. Poor performance on specific subgroups may reveal new preprocessing or feature ideas.
- **Cross-validation**: Use stratified k-fold CV to get robust performance estimates and tune hyperparameters reliably.

### Related Resources:
- [UW-Madison ML-X Nexus — Titanic EDA Notebook](https://colab.research.google.com/github/UW-Madison-DataScience/ML-X-Nexus/blob/main/Learn/Notebooks/Titanic-Dataset.ipynb)
- [Intro to Machine Learning with Sklearn](https://uw-madison-datascience.github.io/ML-X-Nexus/Learn/Workshops/Intro-ML_Sklearn.html)
- [XGBoost Model Guide](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/Models/XGBoost.html)